# StarLayer User Guide (Notebook)

This notebook mirrors the outline in the help guide and is intended to run in Jupyter or Google Colab.

If hosted on GitHub, you can open it in Colab with:

[Open in Colab](https://colab.research.google.com/github/hidden-graph/starlayer/blob/main/docs/user-guide-v1.ipynb)

## How to run this notebook

1. Select the repository virtual environment as your kernel.
2. Run cells from top to bottom so shared variables remain available.
3. If imports fail locally, run `pip install -e .` from the repository root in an activated virtual environment.
4. In Google Colab, run `!pip install git+https://github.com/hidden-graph/starlayer.git` in a new code cell before the rest of the notebook.

In [ ]:
from starlayergraph import StarLayerGraph, Namespace

EX = Namespace("http://example.org/")
RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")

## 1. Graph and literal semantics

- RDF 1.2 triple-term and reification support in the graph model
- Language-direction-aware literal handling, including direction-tagged strings and base-direction semantics

In [ ]:
g = StarLayerGraph()
g.add((EX.claim_7, RDF.reifies, (EX.bob, EX.knows, EX.carol)))
print((EX.claim_7, RDF.reifies, (EX.bob, EX.knows, EX.carol)) in g)

Expected output:

```text
True
```

In [ ]:
g = StarLayerGraph()
g.add((EX.title, EX.value, "مرحبا"))
print(g.serialize(format="turtle12"))

Expected output:

```text
... ex:title ex:value "مرحبا"@ar--rtl .
```

## 2. SPARQL and query semantics

- RDF 1.2-aware SPARQL expression evaluation over triple terms and direction-tagged literals
- Support for RDF 1.2-aware Turtle/SPARQL syntax parsing and serialization
- SPARQL algebra encoded as RDF using a dedicated ontology and SHACL validation schema
- Validation of generated SPARQL RDF representations with SHACL rules
- RDF-graph output for query results and intermediate SPARQL representations

In [ ]:
g = StarLayerGraph()
g.parse(data='''
    @prefix ex: <http://example.org/> .
    ex:note ex:text "مرحبا"@ar--rtl .
''', format='turtle12')
print(g.serialize(format='turtle12'))

Expected output:

```text
@prefix ex: <http://example.org/> .
ex:note ex:text "مرحبا"@ar--rtl .
```

In [ ]:
rows = g.query("""
    PREFIX ex: <http://example.org/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    SELECT ?claim WHERE {
      ?claim rdf:reifies ?statement .
      FILTER( isTRIPLE(?statement) )
    }
""")

for row in rows:
    print(row.claim)

## 3. SHACL validation and rules

- Validation over RDF 1.2 graphs containing triple terms, statement resources, and direction-tagged literals
- SHACL 1.2 node-expression and rule-based evaluation support
- Updated SHACL meta-shapes for SHACL 1.2 compatibility
- Formal SHACL 1.2 UI support for shape-driven interface generation
- Direction-aware uniqueness and datatype constraints for language-tagged values

In [ ]:
from starshacl import StarShaclValidator

data = StarLayerGraph()
data.parse(data='''
    @prefix ex: <http://example.org/> .
    ex:node ex:label "1"@ar--rtl .
''', format='turtle12')

print(data.serialize(format='turtle12'))

Expected output:

```text
@prefix ex: <http://example.org/> .
ex:node ex:label "1"@ar--rtl .
```

## 4. Backend graph-store and format support

- Compatibility with RDF graph backends including Oxigraph, Jena/Fuseki, and SQL-backed stores
- Read/write support for RDF 1.2-aware formats such as Turtle and related RDF serializations
- Dual-mode operation across RDF 1.1 and RDF 1.2 semantics
- Use of backend SPARQL 1.2 capabilities when available, while preserving compatibility with older stores

In [ ]:
g = StarLayerGraph()
g.parse(data='''
    @prefix ex: <http://example.org/> .
    ex:alice ex:claims <<( ex:bob ex:knows ex:carol )>> .
''', format='turtle12')
print(g.serialize(format='turtle12'))

Expected output:

```text
@prefix ex: <http://example.org/> .
ex:alice ex:claims << ex:bob ex:knows ex:carol >> .
```